# 03 · Características textuales para el análisis de frecuencias

## Alcance de esta contribución

Este notebook estudia únicamente las características necesarias para interpretar la frecuencia de palabras y los n-gramas: cantidad de tokens, cantidad de tokens únicos y diversidad léxica. Las características de oraciones, puntuación y otros análisis no se implementan aquí porque pertenecen a otras contribuciones.

La pregunta es si la extensión y variedad del vocabulario se asocian con `content` o `wording`, y si estas medidas cambian entre prompts. No se interpreta ninguna asociación como causalidad.

## 1. Preparación reproducible

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next((path for path in [Path.cwd(), *Path.cwd().parents]
                     if (path / 'src').is_dir() and (path / 'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_csv_files
from src.tokenization import tokenize_text

INTERIM_DATA_DIR = PROJECT_ROOT / 'data' / 'interim'
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for directory in [TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
clean_files = ['summaries_train_clean.csv', 'prompts_train_clean.csv']
datasets = load_csv_files(INTERIM_DATA_DIR, clean_files)
summaries = datasets['summaries_train_clean']
assert summaries['student_id'].is_unique and summaries['text'].notna().all()
assert summaries[['content', 'wording']].notna().all().all()
plt.rcParams.update({'figure.figsize': [10, 4], 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

## 2. Longitud y variedad del vocabulario

Se reutiliza exactamente la tokenización del proyecto. `token_count` mide la extensión; `unique_token_count`, la cantidad de formas diferentes; y `lexical_diversity`, la proporción de tokens distintos. Esta última suele disminuir al crecer un texto, por lo que debe interpretarse junto con la longitud.

In [ ]:
text_features = summaries[['student_id', 'prompt_id', 'content', 'wording']].copy()
text_features['tokens'] = summaries['text'].map(tokenize_text)
text_features['token_count'] = text_features['tokens'].str.len()
text_features['unique_token_count'] = text_features['tokens'].map(lambda values: len(set(values)))
text_features['lexical_diversity'] = text_features['unique_token_count'] / text_features['token_count']
feature_columns = ['token_count', 'unique_token_count', 'lexical_diversity']
assert (text_features['token_count'] > 0).all()
assert np.isfinite(text_features[feature_columns]).all().all()
description = text_features[feature_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
display(description.round(4))
description.to_csv(TABLES_DIR / '03_frequency_feature_descriptive.csv', index_label='variable')
text_features.drop(columns='tokens').to_csv(TABLES_DIR / '03_frequency_features_by_summary.csv', index=False)

## 3. Distribuciones y valores extremos

Los histogramas muestran la forma de cada distribución y los diagramas de caja señalan observaciones alejadas. Los textos largos no se eliminan: son respuestas reales y pueden contener más ocurrencias sólo por su extensión.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=[15, 4])
for ax, column, color in zip(axes, feature_columns, ['#3978a8', '#5b9a72', '#c78145']):
    ax.hist(text_features[column], bins=30, color=color, edgecolor='white')
    ax.axvline(text_features[column].median(), color='#222222', linestyle='--', label='Mediana')
    ax.set(title=column, xlabel='Valor', ylabel='Resúmenes')
    ax.legend()
fig.suptitle('Distribución de características asociadas a frecuencia')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '03_frequency_feature_histograms.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=[15, 4])
for ax, column, color in zip(axes, feature_columns, ['#3978a8', '#5b9a72', '#c78145']):
    ax.boxplot(text_features[column], vert=True, patch_artist=True, boxprops={'facecolor': color, 'alpha': 0.7})
    ax.set(title=column, xticks=[], ylabel='Valor')
fig.suptitle('Dispersión y observaciones extremas')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '03_frequency_feature_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Asociación con `content` y `wording`

Pearson resume relaciones lineales y Spearman relaciones monótonas. Los gráficos conservan todos los puntos y muestran su densidad mediante transparencia.

In [ ]:
analysis_columns = [*feature_columns, 'content', 'wording']
pearson = text_features[analysis_columns].corr(method='pearson').loc[feature_columns, ['content', 'wording']]
spearman = text_features[analysis_columns].corr(method='spearman').loc[feature_columns, ['content', 'wording']]
correlations = pd.concat({'Pearson': pearson, 'Spearman': spearman}, axis=1)
display(correlations.round(4))
correlations.to_csv(TABLES_DIR / '03_frequency_feature_correlations.csv', index_label='variable')

fig, axes = plt.subplots(2, 2, figsize=[11, 8])
for row, feature in enumerate(['token_count', 'lexical_diversity']):
    for column_index, target in enumerate(['content', 'wording']):
        ax = axes[row, column_index]
        ax.scatter(text_features[feature], text_features[target], s=9, alpha=0.18, color='#3978a8')
        ax.set(xlabel=feature, ylabel=target, title=f'{target} frente a {feature}')
fig.suptitle('Características textuales y puntuaciones')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '03_frequency_features_target_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Control por prompt

Se comparan medianas por prompt para detectar si la extensión o variedad dependen del texto fuente. Esto evita atribuir a desempeño una diferencia que en realidad sea temática.

In [ ]:
prompt_profile = (
    text_features.groupby('prompt_id', as_index=False)
    .agg(resumenes=('student_id', 'size'), mediana_tokens=('token_count', 'median'),
         mediana_tokens_unicos=('unique_token_count', 'median'),
         mediana_diversidad=('lexical_diversity', 'median'),
         mediana_content=('content', 'median'), mediana_wording=('wording', 'median'))
)
display(prompt_profile.round(4))
prompt_profile.to_csv(TABLES_DIR / '03_frequency_features_by_prompt.csv', index=False)

## 6. Resultado, interpretación y limitaciones

Las siguientes frases se calculan al ejecutar el notebook. Este análisis sirve de contexto para las frecuencias del Notebook 05: una frecuencia absoluta mayor puede reflejar simplemente un resumen más largo.

In [ ]:
print(f"La mediana es {text_features['token_count'].median():.0f} tokens y {text_features['unique_token_count'].median():.0f} tokens únicos por resumen.")
for feature in feature_columns:
    strongest = correlations.loc[feature, 'Spearman'].abs().idxmax()
    value = correlations.loc[feature, ('Spearman', strongest)]
    print(f"{feature}: la asociación de Spearman de mayor magnitud es con {strongest} ({value:.3f}).")
print(f"Las medianas de longitud por prompt van de {prompt_profile['mediana_tokens'].min():.0f} a {prompt_profile['mediana_tokens'].max():.0f} tokens.")
print('Limitación: la diversidad léxica depende de la longitud y no mide por sí sola la calidad de escritura.')
print('Las asociaciones son exploratorias y no implican causalidad.')

### Hallazgos verificados

Los resúmenes tienen una mediana de 58 tokens y 42 tokens únicos. La longitud presenta una asociación positiva fuerte con `content` (Spearman = 0.850) y moderada con `wording` (0.597). La cantidad de tokens únicos muestra un patrón parecido (0.830 y 0.572, respectivamente). Esto indica asociación, no que escribir más produzca automáticamente una mejor puntuación.

La diversidad léxica se asocia negativamente con `content` (-0.659) y `wording` (-0.488), pero esta medida está matemáticamente condicionada por la longitud: al crecer un texto es más probable repetir palabras. Por ello no debe interpretarse como evidencia de que un vocabulario menos diverso sea mejor.

Las medianas de longitud varían entre 50 y 65 tokens según el prompt. Esta diferencia confirma que las comparaciones de frecuencia deben normalizar por documento y controlar el texto fuente.